# Multi-LoRA email workshop: compare, route, and serve

**Primary question:** How much quality and routing coverage do we trade for fewer specialist calls?

**Time:** 45-60 minutes plus inference. **Prerequisites:** [01](01_email_data_and_routing.ipynb) and [02](02_train_email_adapters.ipynb).

This notebook compares the lexical baseline, base model, generalist LoRA, all three specialists, and text-routed specialists on identical email IDs. It saves resumable predictions and separates local sequential PEFT from optional vLLM multi-adapter serving.

## Evaluation contract

- Default to validation for debugging; switch to the complete test split only after freezing prompts, adapters, and routing thresholds.
- Base and generalist receive the same ten-label prompt. Each specialist receives only its allowed labels. Combine specialist predictions with set union.
- `all-specialists` calls all three domains without using true labels. It is an ablation, **not an oracle**.
- `routed` selects one or more domains from subject/body only. Report skipped-label coverage separately from model quality.
- Invalid JSON, unknown labels, and out-of-domain labels are visible errors, never silently repaired. Transport/model failures halt the run.
- All predictions, including failed formats, remain in the denominator. An invalid call contributes no labels; valid specialist results are retained, with the email marked invalid.
- The local backend switches PEFT adapters sequentially on **one base model**. This demonstrates sharing, not heterogeneous batch throughput.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

import email_workshop
from email_workshop.data import ADAPTERS, load_splits, read_json, write_json
from email_workshop.evaluation import STRATEGIES, binary_confusions, evaluate, lexical_predictions, score
from email_workshop.modeling import LocalBackend, verify_adapters
from email_workshop.routing import fit_router

WORKSHOP_ROOT = Path(email_workshop.__file__).resolve().parents[1]
ARTIFACT_DIR = WORKSHOP_ROOT / Path(os.getenv('EMAIL_WORKSHOP_RUN', 'artifacts/run-2026'))
RUN_EVALUATION = False
RUN_VLLM_DEMO = False
ALLOW_SMOKE_ADAPTERS = False
EVALUATION_SPLIT = 'validation'
EVALUATION_LIMIT = None
BACKEND = 'peft'
DEVICE = 'cuda'
print(f'Evaluation enabled: {RUN_EVALUATION}; backend: {BACKEND}; split: {EVALUATION_SPLIT}')

## Load one shared evaluation set

Do not use different samples for different strategies. A limit is only a pipeline-debugging aid; it is not a stratified quality estimate. Test quality reporting requires `EVALUATION_SPLIT = 'test'` and `EVALUATION_LIMIT = None`.

The router is refitted from the frozen train/validation data, not from this evaluation set. No classifier pickle is loaded.

In [ ]:
splits, data_manifest = load_splits(ARTIFACT_DIR)
if EVALUATION_SPLIT not in ('validation', 'test'):
    raise ValueError('Use validation or test only.')
evaluation_rows = splits[EVALUATION_SPLIT]
if EVALUATION_LIMIT is not None:
    if not 0 < EVALUATION_LIMIT <= len(evaluation_rows):
        raise ValueError('Invalid evaluation limit.')
    evaluation_rows = evaluation_rows[:EVALUATION_LIMIT]
router, router_metadata = fit_router(ARTIFACT_DIR)
print(f'Evaluation emails: {len(evaluation_rows):,}')
if EVALUATION_LIMIT is not None:
    print('DEBUG SUBSET: not a full held-out quality estimate.')

## Verify the backend, then make calls only when enabled

The PEFT backend loads four adapters into one base and uses `disable_adapter()` for the zero-shot baseline. No weights are merged.

The optional vLLM backend accepts loopback endpoints only, checks all model IDs, and uses the same pinned tokenizer and token budget. The operator must verify that the server was started with the printed artifact paths; matching model names alone cannot prove remote weight identity. Optional `VLLM_API_KEY` is read from the environment and never recorded.

In [ ]:
backend = None
if RUN_EVALUATION:
    manifests = verify_adapters(ARTIFACT_DIR)
    if not ALLOW_SMOKE_ADAPTERS and any(item['purpose'] == 'smoke' for item in manifests.values()):
        raise ValueError('Smoke adapters cannot establish quality. Train fully or explicitly allow a smoke comparison.')
    if BACKEND == 'peft':
        backend = LocalBackend(ARTIFACT_DIR, device=DEVICE)
    elif BACKEND == 'vllm':
        from email_workshop.serving import VllmBackend
        backend = VllmBackend(ARTIFACT_DIR)
    else:
        raise ValueError('BACKEND must be peft or vllm.')
else:
    print('Model loading and inference skipped. Set RUN_EVALUATION = True to enable.')

## Run resumable paired inference

Each completed email is atomically checkpointed with its ID, expected/predicted label sets, selected adapters, call count, latency, and explicit errors. Email text and raw generated responses are not saved in prediction files.

The checkpoint fingerprint includes backend/adapters, router configuration, split, and exact row contents. Changing a limit, prompt contract, or adapter requires a new prediction file/run. Interrupted transport failures can resume from the last complete email.

In [ ]:
results = {}
if RUN_EVALUATION:
    results['tfidf'] = lexical_predictions(evaluation_rows, router)
    for strategy in STRATEGIES:
        output = ARTIFACT_DIR / 'evaluation' / f'{BACKEND}-{EVALUATION_SPLIT}-{strategy}.json'
        results[strategy] = evaluate(evaluation_rows, backend, router, router_metadata, strategy, output, EVALUATION_SPLIT)
        print(f'{strategy}: {len(results[strategy])} completed emails')
else:
    print('No predictions fabricated; run inference to populate the comparison.')

## Compare quality, routing coverage, and call counts

Use micro-F1, macro-F1, exact label-set match, Hamming loss, invalid-output rate, and per-label recall. `valid_exact_match` requires both an exact label set and no format failures.

For specialists, routing domain recall measures whether the relevant domains were called. Routing label coverage is an **upper bound on achievable label recall** before specialist errors. It is not measured F1.

Latency is sequential wall-clock latency including routing. It is not warm steady-state server throughput, and model-call count is not a dollar-cost estimate.

In [ ]:
if results:
    expected_ids = [row['id'] for row in evaluation_rows]
    if any([row['id'] for row in rows] != expected_ids for rows in results.values()):
        raise ValueError('Strategies do not cover identical evaluation IDs.')
    summaries, details = {}, {}
    for name, rows in results.items():
        summaries[name], details[name] = score(rows)
    comparison = pd.DataFrame(summaries).T
    display(comparison)
    comparison.to_csv(ARTIFACT_DIR / 'evaluation' / f'{BACKEND}-{EVALUATION_SPLIT}-comparison.csv')
    write_json(ARTIFACT_DIR / 'evaluation' / f'{BACKEND}-{EVALUATION_SPLIT}-metrics.json', {'summary': summaries, 'per_label': details, 'row_count': len(evaluation_rows), 'is_full_split': EVALUATION_LIMIT is None})
    comparison[['micro_f1', 'macro_f1', 'valid_exact_match']].plot.bar(rot=25, ylim=(0, 1), title='Same-row quality comparison')
    plt.tight_layout()
    plt.show()
else:
    print('Metrics unavailable until real inference runs.')

## Inspect finance/travel errors without exposing email text

Look at false negatives, empty outputs, missing routes, and malformed JSON separately. Finance and travel are minority labels: high exact-match accuracy on negative examples is not enough.

Use per-label 2x2 confusion matrices rather than a mutually exclusive ten-way confusion matrix. Inspect text only in your authorized local data workflow, not in shared notebook outputs.

In [ ]:
if results:
    display(pd.concat({name: pd.DataFrame(detail).set_index('label') for name, detail in details.items()}, names=['strategy', 'label']))
    confusion = binary_confusions(results['routed'])
    for label in ['Finance & Bills', 'Travel & Bookings']:
        display(pd.DataFrame(confusion[label], index=['actual no', 'actual yes'], columns=['predicted no', 'predicted yes']).rename_axis(label))
    errors = [row for row in results['routed'] if row['expected'] != row['predicted'] or not row['valid']]
    display(pd.DataFrame(errors).head(10))

## Optional: serve the same adapters with vLLM

Use a **separate Linux/WSL2 GPU environment**, not the Windows PEFT environment. Follow the [serving lab](../README.md#optional-vllm-serving-lab). Record and pin the vLLM release you actually install. Stop/release the notebook's GPU model before starting a second server.

Static `--lora-modules` registration avoids enabling untrusted runtime adapter loading. The generator prints a command; it does not execute it. Set the Linux run path to the same verified artifacts. After server readiness, a concurrent synthetic request demo exercises three different adapters.

Concurrent successful requests do not prove they shared a GPU batch. Inspect server telemetry before claiming heterogeneous batching performance.

In [ ]:
import shlex
from email_workshop.serving import vllm_command

LINUX_RUN_ROOT = 'artifacts/run-2026'
if all((ARTIFACT_DIR / 'adapters' / name / 'manifest.json').exists() for name in ADAPTERS):
    print(shlex.join(vllm_command(ARTIFACT_DIR, LINUX_RUN_ROOT)))
else:
    print('Train all four adapters before generating the server command.')

In [ ]:
if RUN_VLLM_DEMO:
    from email_workshop.serving import VllmBackend, concurrent_demo
    serving_backend = VllmBackend(ARTIFACT_DIR)
    display(pd.DataFrame(concurrent_demo(serving_backend, ARTIFACT_DIR / 'evaluation' / 'vllm-concurrent-smoke.json')))
else:
    print('vLLM calls skipped. Enable only after explicitly starting and verifying the local server.')

## Interpretation and final test gate

1. Freeze adapters, prompts, label threshold, and routing threshold using validation only.
2. Set the full test split; run every strategy on exactly the same IDs.
3. Compare the generalist against all-specialists to isolate specialization from routing.
4. Compare all-specialists against routed to isolate routing losses.
5. Report finance and travel recall, invalid outputs, and routing coverage alongside aggregate F1.
6. Retain all failed-format rows. Never replace a failed adapter with the base model without explicitly measuring a different strategy.
7. Do not claim multi-LoRA wins: report the observed result, including regressions.
8. Treat this small, possibly templated dataset as a teaching benchmark. Exact deduplication does not remove near-template leakage; validate on independent, consented enterprise data before production.
9. This workshop does not establish Nemotron/NIM or Foundry managed-adapter support. Reuse the curriculum's compatibility gates before changing runtime/model.